In [ ]:
!pip install panda-gym stable-baselines3 gymnasium

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.5/80.5 MB 9.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.5/187.5 kB 14.7 MB/s eta 0:00:00
  Created wheel for pybullet: filename=pybullet-3.2.7-cp312-cp312-linux_x86_64.whl size=99873173 sha256=43cddc24b44b62d20c737b27d4868e8a45934cca16f91bbaff394920702d3932
  Stored in directory: /root/.cache/pip/wheels/72/95/1d/b336e5ee612ae9a019bfff4dc0bedd100ee6f0570db205fdf8
Successfully built pybullet


In [ ]:
import gymnasium as gym
import panda_gym
import numpy as np

from stable_baselines3 import PPO, SAC, HER
from stable_baselines3.her import HerReplayBuffer

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
env = gym.make("PandaPush-v3", render_mode="rgb_array")

obs, info = env.reset()
print("Observation keys:", obs.keys())
print("Environment OK ✅")

Observation keys: dict_keys(['observation', 'achieved_goal', 'desired_goal'])
Environment OK ✅


In [ ]:
env = gym.make("PandaPush-v3", render_mode="rgb_array")

model_ppo = PPO(
    policy="MultiInputPolicy",
    env=env,
    verbose=1,
    learning_rate=3e-4,
    n_steps=2048,
    batch_size=64,
    gamma=0.99,
)

model_ppo.learn(total_timesteps=300000)

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 41.5     |
|    ep_rew_mean     | -41.3    |
|    success_rate    | 0.184    |
| time/              |          |
|    fps             | 236      |
|    iterations      | 1        |
|    time_elapsed    | 8        |
|    total_timesteps | 2048     |
---------------------------------


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 45          |
|    ep_rew_mean          | -44.9       |
|    success_rate         | 0.11        |
| time/                   |             |
|    fps                  | 212         |
|    iterations           | 2           |
|    time_elapsed         | 19          |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.004706111 |
|    clip_fraction        | 0.0219      |
|    clip_range           | 0.2         |
|    entropy_loss         | -4.25       |
|    explained_variance   | 0.00619     |
|    learning_rate        | 0.0003      |
|    loss                 | 11.9        |
|    n_updates            | 10          |
|    policy_gradient_loss | -0.00542    |
|    std                  | 0.997       |
|    value_loss           | 95.5        |
-----------------------------------------
----------------------------------

In [ ]:
model_ppo.save("ppo_panda_push")

In [ ]:
env = gym.make("PandaPush-v3", render_mode="rgb_array")

model_ppo = PPO.load("ppo_panda_push")

episode_returns = []

for ep in range(10):
    obs, _ = env.reset()
    done = False
    total_reward = 0

    while not done:
        action, _ = model_ppo.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, _ = env.step(action)
        total_reward += reward
        done = terminated or truncated

    episode_returns.append(total_reward)
    print(f"PPO Episode {ep+1}: return = {total_reward:.2f}")

print("Mean return:", np.mean(episode_returns))

PPO Episode 1: return = -50.00
PPO Episode 2: return = -50.00
PPO Episode 3: return = -50.00
PPO Episode 4: return = -50.00
PPO Episode 5: return = -50.00
PPO Episode 6: return = -50.00
PPO Episode 7: return = -50.00
PPO Episode 8: return = -50.00
PPO Episode 9: return = -50.00
PPO Episode 10: return = -50.00
Mean return: -50.0


In [ ]:
env = gym.make("PandaPush-v3", render_mode="rgb_array")

model_sac = SAC(
    "MultiInputPolicy",
    env,
    replay_buffer_class=HerReplayBuffer,
    replay_buffer_kwargs=dict(
        n_sampled_goal=4,
        goal_selection_strategy="future",
    ),
    verbose=1,
    buffer_size=1_000_000,
    learning_rate=3e-4,
    batch_size=256,
    gamma=0.98,
)

model_sac.learn(total_timesteps=300000)

Output streaming troncato alle ultime 5000 righe.
|    success_rate    | 0.5      |
| time/              |          |
|    episodes        | 6800     |
|    fps             | 26       |
|    time_elapsed    | 10123    |
|    total_timesteps | 267108   |
| train/             |          |
|    actor_loss      | 4.73     |
|    critic_loss     | 2.7      |
|    ent_coef        | 0.0182   |
|    ent_coef_loss   | -0.55    |
|    learning_rate   | 0.0003   |
|    n_updates       | 267007   |
---------------------------------
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 33.2     |
|    ep_rew_mean     | -32.8    |
|    success_rate    | 0.49     |
| time/              |          |
|    episodes        | 6804     |
|    fps             | 26       |
|    time_elapsed    | 10130    |
|    total_timesteps | 267306   |
| train/             |          |
|    actor_loss      | 4.3      |
|    critic_loss     | 4.03     |
|    ent_coef        | 0.0183   

In [ ]:
model_sac.save("sac_panda_push")

In [ ]:
env = gym.make("PandaPush-v3", render_mode="rgb_array")

model_sac = SAC.load("sac_panda_push", env=env)

episode_returns = []

for ep in range(100):
    obs, _ = env.reset()
    done = False
    total_reward = 0

    while not done:
        action, _ = model_sac.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, _ = env.step(action)
        total_reward += reward
        done = terminated or truncated

    episode_returns.append(total_reward)
    print(f"SAC Episode {ep+1}: return = {total_reward:.2f}")

print("Mean return:", np.mean(episode_returns))

Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
SAC Episode 1: return = -50.00
SAC Episode 2: return = -19.00
SAC Episode 3: return = -8.00
SAC Episode 4: return = -8.00
SAC Episode 5: return = -50.00
SAC Episode 6: return = -50.00
SAC Episode 7: return = -50.00
SAC Episode 8: return = -18.00
SAC Episode 9: return = 0.00
SAC Episode 10: return = -8.00
SAC Episode 11: return = -8.00
SAC Episode 12: return = -8.00
SAC Episode 13: return = -14.00
SAC Episode 14: return = -50.00
SAC Episode 15: return = -50.00
SAC Episode 16: return = -8.00
SAC Episode 17: return = -15.00
SAC Episode 18: return = -12.00
SAC Episode 19: return = -4.00
SAC Episode 20: return = -8.00
SAC Episode 21: return = -50.00
SAC Episode 22: return = -13.00
SAC Episode 23: return = -8.00
SAC Episode 24: return = -17.00
SAC Episode 25: return = -50.00
SAC Episode 26: return = -50.00
SAC Episode 27: return = -50.00
SAC Episode 28: return = -9.00
SAC Episode 29: return = -50.00
SAC Episode 30: 